# 진짜 최종 구현 코드


In [1]:
# 1. Load

import ast
import numpy as np
import pandas as pd

# 원본 세션 테이블
promotion = pd.read_csv('./data/promotion_df.csv')

# transaction 재계산용 원본
merged = pd.read_csv('./data/all_merged.csv')


In [2]:
# 2. transaction 누적합 준비

# transaction만 뽑아서 사람별 누적합을 만든다
transactions = merged[merged['event'].eq('transaction')].copy()
transactions = transactions[['person', 'time', 'amount', 'Unnamed: 0']].sort_values(
    ['person', 'time', 'Unnamed: 0']
).reset_index(drop=True)

transactions['cum_amount'] = transactions.groupby('person')['amount'].cumsum()

# 사람별 조회용 lookup
tx_lookup = {
    person: (
        g['time'].to_numpy(dtype=float),
        g['cum_amount'].to_numpy(dtype=float)
    )
    for person, g in transactions.groupby('person', sort=False)
}

# start ~ end 사이의 거래 합을 구하는 함수
def amount_between(person, start, end):
    if pd.isna(start) or pd.isna(end):
        return np.nan

    item = tx_lookup.get(person)
    if item is None:
        return np.nan

    times, cum = item
    end_idx = np.searchsorted(times, float(end), side='right') - 1
    start_idx = np.searchsorted(times, float(start), side='left') - 1

    end_cum = float(cum[end_idx]) if end_idx >= 0 else 0.0
    start_cum = float(cum[start_idx]) if start_idx >= 0 else 0.0

    return round(end_cum - start_cum, 2)


In [3]:
# 3. 76건 보정 로직 만들기

# 작업용 테이블
promo_seq = promotion.sort_values(['person', 'offer_id', 'offer received', 'offer completed']).copy()

# 같은 person + offer_id 안에서 이전 received 찾기
promo_seq['prev_received'] = promo_seq.groupby(['person', 'offer_id'])['offer received'].shift(1)

# 1차 로직: 현재 received 기준 amount
promo_seq['completion_amount_current'] = promo_seq.apply(
    lambda r: amount_between(r['person'], r['offer received'], r['offer completed']),
    axis=1
)

# 2차 로직: 이전 received 기준 amount
promo_seq['completion_amount_prev_received'] = promo_seq.apply(
    lambda r: amount_between(r['person'], r['prev_received'], r['offer completed']),
    axis=1
)

# 현재 기준으로는 difficulty 미만인데, 이전 received로 보면 difficulty를 넘는 74건
candidate_fixable = promo_seq[
    (promo_seq['is_completed'].eq(1)) &
    (promo_seq['is_normal_flow'].eq(1)) &
    (promo_seq['completion_amount_current'].lt(promo_seq['difficulty'])) &
    (promo_seq['completion_amount_prev_received'].ge(promo_seq['difficulty']))
].copy()

# 현재 기준으로는 difficulty 미만인 anomaly들
anomaly = promo_seq[
    (promo_seq['is_completed'].eq(1)) &
    (promo_seq['is_normal_flow'].eq(1)) &
    (promo_seq['completion_amount_current'].lt(promo_seq['difficulty']))
].copy()

# 2건 block 로직용 요약
block_df = (
    anomaly.groupby(['person', 'offer_id'], as_index=False)
    .agg(
        rows=('offer_cycle', 'size'),
        block_start=('offer received', 'min'),
        block_end=('offer completed', 'max'),
        difficulty=('difficulty', 'first')
    )
)

# 같은 person + offer_id가 2행 이상인 그룹만 block으로 본다
block_df = block_df[block_df['rows'] > 1].copy()

# block 전체 구간의 총 amount
block_df['block_completion_amount'] = block_df.apply(
    lambda r: amount_between(r['person'], r['block_start'], r['block_end']),
    axis=1
)

# 최종값은 current로 시작
promo_seq['adjusted_completion_amount'] = promo_seq['completion_amount_current']
promo_seq['count_in_total'] = 1
promo_seq['adjustment_type'] = 'current'

# 74건은 prev_received 기준으로 교체
promo_seq.loc[candidate_fixable.index, 'adjusted_completion_amount'] = \
    promo_seq.loc[candidate_fixable.index, 'completion_amount_prev_received']
promo_seq.loc[candidate_fixable.index, 'adjustment_type'] = 'prev_received'


In [4]:
# 4. 2건 block 반영 + 최종 DF 만들기

# block 그룹은 전체 group을 한 번만 세도록 대표행 1개만 남긴다

for _, row in block_df.iterrows():
    person = row['person']
    offer_id = row['offer_id']
    block_start = row['block_start']
    block_amount = row['block_completion_amount']

    # 이 블록에 속한 행들
    block_idx = promo_seq[
        promo_seq['person'].eq(person) &
        promo_seq['offer_id'].eq(offer_id)
    ].index

    # block_start와 정확히 같은 received를 가진 행을 대표행으로 선정
    leader_candidates = promo_seq[
        promo_seq['person'].eq(person) &
        promo_seq['offer_id'].eq(offer_id) &
        promo_seq['offer received'].eq(block_start)
    ].index

    leader_idx = leader_candidates[0]

    # 블록에 속한 행들은 제외
    promo_seq.loc[block_idx, 'adjusted_completion_amount'] = 0
    promo_seq.loc[block_idx, 'count_in_total'] = 0
    promo_seq.loc[block_idx, 'adjustment_type'] = 'block_other'

    # 대표행만 블록 총액 반영
    promo_seq.loc[leader_idx, 'adjusted_completion_amount'] = block_amount
    promo_seq.loc[leader_idx, 'count_in_total'] = 1
    promo_seq.loc[leader_idx, 'adjustment_type'] = 'block_leader'




# 최종 amount 총합
total_adjusted_amount = promo_seq.loc[
    promo_seq['count_in_total'].eq(1),
    'adjusted_completion_amount'
].sum()

print('최종 amount 총합:', total_adjusted_amount)

# 최종 df 생성
promotion_final = promotion.copy()

# 원본 amount 보존
promotion_final['amount_raw'] = promotion_final['amount']

# person + offer_id + offer_cycle 기준으로 최종 amount 붙이기
promotion_final = promotion_final.merge(
    promo_seq[['person', 'offer_id', 'offer_cycle', 'adjusted_completion_amount']],
    on=['person', 'offer_id', 'offer_cycle'],
    how='left'
)

# 최종 분석용 amount 반영
promotion_final['amount'] = promotion_final['adjusted_completion_amount']


최종 amount 총합: 745929.74


In [5]:
# 5. 최종 결과 확인
display(
    promotion_final[
        [
            'person', 'offer_id', 'offer_cycle',
            'offer received', 'offer viewed', 'offer completed',
            'amount_raw', 'adjusted_completion_amount', 'amount',
            'difficulty'
        ]
    ].head(20)
)

# 아직 difficulty보다 작은 completed가 남았는지 확인
remain = promotion_final[
    (promotion_final['is_completed'].eq(1)) &
    (promotion_final['is_normal_flow'].eq(1)) &
    (promotion_final['amount'] < promotion_final['difficulty'])
].copy()

print('여전히 difficulty보다 작은 정상 completed 건수:', len(remain))


,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,amount_raw,adjusted_completion_amount,amount,difficulty
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,8.57,8.57,8.57,5
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,14.11,14.11,14.11,10
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,10.27,10.27,10.27,10
3,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,11.93,11.93,11.93,7
4,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,22.05,22.05,22.05,5
5,0011e0d4e6b944f998e987f904e8c1e5,discount_20_5_10,Discount_1,408.0,432.0,576.0,22.05,22.05,22.05,20
6,0020c2b971eb4e9188eac86d93036a77,bogo_10_10_5,Bogo_1,408.0,426.0,510.0,17.24,17.24,17.24,10
7,0020c2b971eb4e9188eac86d93036a77,discount_10_2_10,Discount_2,336.0,NaN,510.0,17.24,17.24,17.24,10
8,0020c2b971eb4e9188eac86d93036a77,discount_10_2_10,Discount_1,0.0,12.0,54.0,17.63,17.63,17.63,10
9,0020ccbbb6d84e358d3414a3ff76cffd,discount_7_3_7,Discount_1,168.0,168.0,222.0,11.65,11.65,11.65,7


여전히 difficulty보다 작은 정상 completed 건수: 2


In [6]:
# 최종 저장용 정리
promotion_final = promotion_final.drop(columns=['adjusted_completion_amount'])


In [7]:
# 6. 저장

promotion_final.to_csv('./data/promotion_final.csv', index=False)


# 정상흐름 amount 구해보기


In [8]:
# 수정 전 정상흐름 amount 합
normal_total_amount_all2 = promotion_final.loc[
    promotion_final['is_deduplicated'].eq(1),
    'amount_raw'
].sum()

# 수정 후 정상흐름 amount 합
normal_total_amount_all = promotion_final.loc[
    promotion_final['is_deduplicated'].eq(1),
    'amount'
].sum()


print("수정 전 정상흐름 amount 합", normal_total_amount_all2)
print("수정 후 정상흐름 amount 합", normal_total_amount_all)
print("두 amount 차이", round(normal_total_amount_all - normal_total_amount_all2, 2))



수정 전 정상흐름 amount 합 442993.24
수정 후 정상흐름 amount 합 486621.87
두 amount 차이 43628.63


## `amount` 재계산 로직 정리

이 노트북에서는 프로모션 완료 시점에 붙어 있던 기존 `amount`를 그대로 쓰지 않고,  
`transcript.csv`의 거래 흐름을 다시 따라가서 더 자연스러운 최종 `amount`를 계산했다.

### 1. 기본 원리
- `offer received` 이후부터 `offer completed` 이전까지의 거래 금액을 합쳐 `completion_amount_current`를 계산했다.
- `offer viewed` 이후부터 `offer completed` 이전까지의 거래 금액도 따로 계산해 `promo_influenced_amount`로 확인했다.
- 같은 `person + offer_id` 안에서 이전 `received`를 시작점으로 다시 계산한 값은 `completion_amount_prev_received`로 만들었다.

### 2. 76건의 이상치 처리
- `completion_amount_current < difficulty`인데도 `is_completed == 1`이고 `is_normal_flow == 1`인 행들을 이상치로 잡았다.
- 이 중 대부분(74건)은 현재 `received`가 너무 뒤에 잡혀서 앞쪽 amount가 빠진 케이스였다.
- 그래서 `prev_received`를 시작점으로 다시 계산했을 때 `difficulty`를 넘는 행들은 `completion_amount_prev_received`를 최종값으로 사용했다.

### 3. 2건의 block 처리
- 남은 특수 케이스 2건은 같은 `person + offer_id`의 연속 회차를 하나의 block으로 묶어서 다시 계산했다.
- 이때 block의 시작은 `block_start = 가장 이른 offer received`, block의 끝은 `block_end = 가장 늦은 offer completed`로 잡았다.
- 그 구간의 거래 합을 `block_completion_amount`로 계산했다.
- block 안에서는 중복 합산을 막기 위해 대표행 `block_leader` 1개만 `count_in_total = 1`로 두고, 나머지 행은 `count_in_total = 0`으로 제외했다.

### 4. 최종 amount 구성
- 최종적으로 `adjusted_completion_amount`를 만들고,
- 이를 `promotion_final['amount']`에 반영했다.
- 원래 값은 `amount_raw`로 따로 보존했다.

### 5. 최종 합계 기준
- 최종 총합은 `count_in_total == 1`인 행만 더해서 구했다.
- 마지막 합계는 block 중복을 제외한 상태로 계산되었다.
- 최종 총합을 볼 때는 `is_normal_flow`보다 `is_deduplicated` 기준이 더 적절하다.
  - `is_normal_flow`는 흐름이 정상인지 보는 플래그
  - `is_deduplicated`는 정상 흐름 중 더블카운팅을 제외한 최종 합산 기준이다

### 6. 최종 결과
- `promotion_final`은 원본 `promotion_df`에 최종 `amount`가 반영된 최종 데이터프레임이다.
- `amount_raw`는 원래 값, `amount`는 최종 분석용 값이다.
